# Progres 1 — Pengumpulan dan Persiapan Data
**Proyek:** Peramalan Realisasi BPHTB Kota Surabaya Menggunakan Metode SARIMA dan Random Forest

In [ ]:
import os

def pastikan_file(nama):
    """Hentikan notebook dengan pesan jelas jika file input belum ada."""
    if not os.path.exists(nama):
        raise FileNotFoundError(
            f"File '{nama}' tidak ditemukan di folder {os.getcwd()}. "
            "Pastikan file ada di folder yang sama dengan notebook, "
            "atau jalankan progres sebelumnya terlebih dahulu.")

def unduh(*nama_file):
    """Di VS Code file sudah tersimpan di folder kerja; cukup tampilkan daftarnya."""
    for f in nama_file:
        print("Tersimpan:", os.path.abspath(f))

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)

## 1.1 Membaca file sumber
File Excel berisi satu sheet `uraian` dengan blok kolom per tahun (Target, SSPD, Realisasi, %).

In [ ]:
NAMA_FILE = "Data_Pendukung_BPHTB_PAK_2026_dan_Target_2027.xlsx"
pastikan_file(NAMA_FILE)
raw = pd.read_excel(NAMA_FILE, sheet_name="uraian", header=None)
print("Ukuran sheet:", raw.shape)
raw.iloc[:18, :6]

## 1.2 Memetakan posisi kolom
Baris 5–16 berisi Januari–Desember, baris 17 berisi Total. Kolom realisasi tiap tahun:

| Tahun | Kolom target | Kolom realisasi | Keterangan |
|---|---|---|---|
| 2021–2024 | 2, 6, 10, 14 | 4, 8, 12, 16 | Realisasi |
| 2025 | 18 | 24 | Total Realisasi (Jual Beli + Non Jual Beli) |
| 2026 | 26 | 32 | Total Realisasi; Jan–Agu riil, Sep–Des estimasi |

In [ ]:
BARIS_BULAN = list(range(5, 17))
BARIS_TOTAL = 17
KOL_REALISASI = {2021: 4, 2022: 8, 2023: 12, 2024: 16, 2025: 24, 2026: 32}
KOL_TARGET    = {2021: 2, 2022: 6, 2023: 10, 2024: 14, 2025: 18, 2026: 26}

# cek label kolom agar pemetaan tidak salah
def judul_terdekat(baris, k):
    """Ambil label terisi terdekat di kiri kolom k (judul blok tahun memakai sel gabungan)."""
    for j in range(k, -1, -1):
        if pd.notna(raw.iloc[baris, j]):
            return raw.iloc[baris, j]
for th, k in KOL_REALISASI.items():
    print(th, "| blok:", judul_terdekat(2, k), "| kolom:", raw.iloc[4, k], "| Januari:", f"{raw.iloc[5, k]:,.0f}")

## 1.3 Ekstraksi ke deret waktu bulanan

In [ ]:
nilai = []
for th, k in KOL_REALISASI.items():
    v = raw.loc[BARIS_BULAN, k].astype(float).values
    nilai.extend(v[:8] if th == 2026 else v)          # 2026: hanya Jan–Agu yang riil

y = pd.Series(np.array(nilai) / 1e9,
              index=pd.date_range("2021-01-01", periods=len(nilai), freq="MS"),
              name="realisasi_miliar")
y.index.name = "bulan"

estimasi_bapenda = pd.Series(raw.loc[BARIS_BULAN[8:], 32].astype(float).values / 1e9,
                             index=pd.date_range("2026-09-01", periods=4, freq="MS"),
                             name="estimasi_bapenda_miliar")
estimasi_bapenda.index.name = "bulan"

target = pd.Series({th: float(raw.loc[BARIS_TOTAL, k]) / 1e9 for th, k in KOL_TARGET.items()},
                   name="target_miliar")
target.index.name = "tahun"
y.head(14)

## 1.4 Validasi data

In [ ]:
realisasi_th = y[:"2025"].groupby(y[:"2025"].index.year).sum()
total_excel = pd.Series({th: float(raw.loc[BARIS_TOTAL, KOL_REALISASI[th]]) / 1e9 for th in range(2021, 2026)})
validasi = pd.DataFrame({"Total hasil ekstraksi": realisasi_th.values, "Total di Excel": total_excel.values},
                        index=range(2021, 2026))
validasi["Selisih"] = validasi.iloc[:, 0] - validasi.iloc[:, 1]
print(validasi.round(3))
assert np.allclose(validasi["Selisih"], 0, atol=1e-6), "Ekstraksi tidak cocok dengan Excel!"

print("\nNilai hilang:", y.isna().sum(), "| Nilai <= 0:", (y <= 0).sum())
print("Frekuensi lengkap tanpa bulan bolong:", len(y) == len(pd.date_range(y.index[0], y.index[-1], freq="MS")))

## 1.5 Menyimpan dataset bersih

In [ ]:
y.to_csv("bphtb_bulanan.csv")
estimasi_bapenda.to_csv("estimasi_bapenda_sep_des_2026.csv")
target.to_csv("target_bphtb.csv")
print(pd.concat([target.rename("Target (M)"),
                 y.groupby(y.index.year).sum().rename("Realisasi (M)")], axis=1).round(1))
unduh("bphtb_bulanan.csv", "estimasi_bapenda_sep_des_2026.csv", "target_bphtb.csv")